In [95]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico - DC.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico - DC.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico - DC.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [96]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y      rotation
count  12860.000000   12860.000000  12860.000000
mean   35060.961230  128079.732587     25.250389
std      593.173520     951.762144     62.562695
min    33667.553332  125996.367175      0.000000
25%    34622.695128  127293.973600      0.000000
50%    35000.922088  128002.283775      0.000000
75%    35483.685992  128959.993661      0.000000
max    36625.891051  129787.710488    270.000000
<class 'pandas.DataFrame'>
RangeIndex: 12889 entries, 0 to 12888
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      12889 non-null  str    
 1   layer     12889 non-null  str    
 2   handle    12889 non-null  str    
 3   name      12860 non-null  str    
 4   X         12860 non-null  float64
 5   Y         12860 non-null  float64
 6   rotation  12860 non-null  float64
 7   UFV       12860 non-null  str    
 8   PST       12860 non-null  str    
 9   INV       12831 non-null  s

In [97]:
string_mask = (ELM_df['name'] == "1x96p_ponta") | (ELM_df['name'] == "1x96p_meio") | (ELM_df['name'] == "1x64p") | (ELM_df['name'] == "1x32p")

strings_df = ELM_df[string_mask].copy()

strings_df.rename(columns={'X': 'Position_X', 'Y': 'Position_Y'}, inplace=True)
strings_df.drop(columns=['rotation', 'type', 'handle', 'layer', 'QBT', 'DSJ', 'PVM'], inplace=True)
strings_df.to_excel(r"C:\Users\Usuario\Desktop\teste2.xlsx")
strings_df['UFV'] = strings_df['UFV'].astype(int)
strings_df['PST'] = strings_df['PST'].astype(int)
strings_df['INV'] = strings_df['INV'].astype(int)
strings_df['Line'] = round(strings_df['Position_X'],0)
print(strings_df.describe())
print(strings_df.info())
print(strings_df)


         Position_X     Position_Y      UFV           PST           INV  \
count  12367.000000   12367.000000  12367.0  12367.000000  12367.000000   
mean   35060.913921  128079.153417      0.0     15.012857      8.496159   
std      593.266859     951.882477      0.0      8.364836      4.608246   
min    33667.553332  126018.799500      0.0      1.000000      1.000000   
25%    34622.695128  127293.973600      0.0      8.000000      4.000000   
50%    35003.257868  128002.283775      0.0     15.000000      9.000000   
75%    35483.685992  128959.993661      0.0     22.000000     12.000000   
max    36625.891051  129787.710488      0.0     29.000000     16.000000   

               Line  
count  12367.000000  
mean   35060.886310  
std      593.265156  
min    33668.000000  
25%    34623.000000  
50%    35003.000000  
75%    35484.000000  
max    36626.000000  
<class 'pandas.DataFrame'>
Index: 12367 entries, 58 to 12888
Data columns (total 8 columns):
 #   Column      Non-Null Count  

In [98]:
inverter_mask = (ELM_df['name'] == "EMF397_inversor")

inverter_df = ELM_df[inverter_mask].copy()
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer'], inplace=True)
inverter_df["GRP"] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str).str.zfill(2) + "." + inverter_df["INV"].astype(str).str.zfill(2)
inverter_df['inverter_line'] = round(inverter_df['inverter_x'],0)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

         inverter_x     inverter_y  inverter_line
count    464.000000     464.000000     464.000000
mean   35060.860731  128094.675668   35060.872845
std      593.508362     951.375667     593.495301
min    33680.175772  125996.367175   33680.000000
25%    34625.333313  127271.528300   34625.500000
50%    34994.633150  128067.144800   34994.500000
75%    35477.408313  128996.151570   35477.000000
max    36569.168916  129708.243463   36569.000000
<class 'pandas.DataFrame'>
Index: 464 entries, 7584 to 7150
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   inverter_x     464 non-null    float64
 1   inverter_y     464 non-null    float64
 2   UFV            464 non-null    str    
 3   PST            464 non-null    str    
 4   INV            464 non-null    str    
 5   QBT            464 non-null    str    
 6   DSJ            464 non-null    str    
 7   PVM            464 non-null    str    
 8   GRP         

In [99]:
strings_df["GRP"] = strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str).str.zfill(2) + "." + strings_df["INV"].astype(str).str.zfill(2)
strings_df.sort_values(by=['Line', 'Position_Y'], ascending=[True, False], inplace=True)


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Different PSTs
condition2 = strings_df["GRP"] != strings_df["GRP"].shift()

#Break in the Structure
condition3 = abs(strings_df["Position_Y"] - strings_df["Position_Y"].shift()) > 44

break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition3

strings_df['SubGroup'] = break_condition_1.cumsum() # The SubGroup represents strings that are on the same Line, are from the same inverter and don't have a break between them. Used to count the number of strings that are going down the pile in the end.
strings_df['Group'] = break_condition_2.cumsum() # The Group represents strings that are on the same Line and don't have a break between them. Used to calculate where the start and end piles are.
print(strings_df)
strings_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

              name    Position_X     Position_Y  UFV  PST  INV      GRP  \
12074        1x32p  33667.553648  129034.804861    0    7    1  0.07.01   
9730   1x96p_ponta  33667.553490  128974.069070    0    7    1  0.07.01   
9653    1x96p_meio  33667.553490  128931.290545    0    7    1  0.07.01   
9729   1x96p_ponta  33667.553490  128888.512020    0    7    1  0.07.01   
9726         1x64p  33667.553490  128845.246495    0    7    1  0.07.01   
...            ...           ...            ...  ...  ...  ...      ...   
11538        1x32p  36600.691051  129536.319495    0    3    3  0.03.03   
11888        1x32p  36606.991051  129536.319495    0    3    3  0.03.03   
12388        1x32p  36613.291051  129536.319495    0    3    3  0.03.03   
12389        1x32p  36619.591051  129536.319495    0    3    3  0.03.03   
12390        1x32p  36625.891051  129536.319495    0    3    3  0.03.03   

          Line  SubGroup  Group  
12074  33668.0         1      1  
9730   33668.0         2      2

In [100]:
distance_32p = 20.1855 # Distance between the middle point of the string and the 32p tracker's pile.
distance_64p = 20.1985 # Distance between the middle point of the string and the 64p tracker's pile.
distance_96p_meio = 20.1855 # Distance between the middle point of the string and the 96p_meio tracker's pile.
distance_96p_ponta = 19.935 # Distance between the middle point of the string and the 96p_ponta tracker's pile.

strings_df['N exit'] = np.where(
    strings_df['name'] == '1x32p',
    strings_df['Position_Y'] + distance_32p,
    np.where(
        strings_df['name'] == '1x64p',
        strings_df['Position_Y'] + distance_64p,
        np.where(
            strings_df['name'] == '1x96p_meio',
            strings_df['Position_Y'] + distance_96p_meio,
            strings_df['Position_Y'] + distance_96p_ponta,
        )
    )
)

strings_df['S exit'] = np.where(
    strings_df['name'] == '1x32p',
    strings_df['Position_Y'] - distance_32p,
    np.where(
        strings_df['name'] == '1x64p',
        strings_df['Position_Y'] - distance_64p,
        np.where(
            strings_df['name'] == '1x96p_meio',
            strings_df['Position_Y'] - distance_96p_meio,
            strings_df['Position_Y'] - distance_96p_ponta,
        )
    )
)


strings_df['SubGroup - Position Y - N'] = strings_df.groupby("SubGroup")["N exit"].transform('max')
strings_df['SubGroup - Position Y - S'] = strings_df.groupby("SubGroup")["S exit"].transform('min')

strings_df['Group - Position Y - N'] = strings_df.groupby("Group")["N exit"].transform('max')
strings_df['Group - Position Y - S'] = strings_df.groupby("Group")["S exit"].transform('min')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['Line'].transform('count')

print(strings_df)

              name    Position_X     Position_Y  UFV  PST  INV      GRP  \
12074        1x32p  33667.553648  129034.804861    0    7    1  0.07.01   
9730   1x96p_ponta  33667.553490  128974.069070    0    7    1  0.07.01   
9653    1x96p_meio  33667.553490  128931.290545    0    7    1  0.07.01   
9729   1x96p_ponta  33667.553490  128888.512020    0    7    1  0.07.01   
9726         1x64p  33667.553490  128845.246495    0    7    1  0.07.01   
...            ...           ...            ...  ...  ...  ...      ...   
11538        1x32p  36600.691051  129536.319495    0    3    3  0.03.03   
11888        1x32p  36606.991051  129536.319495    0    3    3  0.03.03   
12388        1x32p  36613.291051  129536.319495    0    3    3  0.03.03   
12389        1x32p  36619.591051  129536.319495    0    3    3  0.03.03   
12390        1x32p  36625.891051  129536.319495    0    3    3  0.03.03   

          Line  SubGroup  Group         N exit         S exit  \
12074  33668.0         1      1  1

In [101]:
groups_df = strings_df.groupby("SubGroup")[['GRP','Position_X', 'Position_Y', 'SubGroup - Position Y - N', 'SubGroup - Position Y - S', 'Group - Position Y - N', 'Group - Position Y - S', 'No. of Strings', 'Line']].first()
groups_df.rename(columns={'Position_X': 'X', 'Position_Y': 'Y', 'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 8

              GRP             X              Y        Y-N-STR        Y-S-STR  \
SubGroup                                                                       
1         0.07.01  33667.553648  129034.804861  129054.990361  129014.619361   
2         0.07.01  33667.553490  128974.069070  128994.004070  128782.545995   
3         0.07.01  33673.853648  129034.804861  129054.990361  129014.619361   
4         0.07.01  33673.853490  128974.069070  128994.004070  128782.545995   
5         0.07.01  33680.153648  129034.804861  129054.990361  129014.619361   
...           ...           ...            ...            ...            ...   
4670      0.03.03  36600.691051  129536.319495  129556.504995  129516.133995   
4671      0.03.03  36606.991051  129536.319495  129556.504995  129516.133995   
4672      0.03.03  36613.291051  129536.319495  129556.504995  129516.133995   
4673      0.03.03  36619.591051  129536.319495  129556.504995  129516.133995   

In [102]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['N or S or M'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) > abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
    "S",
    np.where(
        (groups_df['inverter_y'] < groups_df['Y-N-TRK']) & (groups_df['inverter_y'] > groups_df['Y-S-TRK']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
groups_df['closest_y'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) < abs(groups_df['Y'] - groups_df['inverter_y']),
    groups_df['Y-N-TRK'],
    np.where(
        abs(groups_df['Y'] - groups_df['inverter_y']) < abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
        groups_df['Y'],
        groups_df['Y-S-TRK']
    )
)

groups_df.sort_values(['GRP', 'Line'], inplace = True)

print(groups_df.head(20))


          GRP            X              Y        Y-N-STR        Y-S-STR  \
4173  0.01.01  35878.30525  129697.804096  129717.989596  129677.618596   
4178  0.01.01  35884.60525  129701.771663  129721.957163  129681.586163   
4183  0.01.01  35890.90525  129705.739231  129725.924731  129685.553731   
4189  0.01.01  35897.20525  129709.706799  129729.892299  129689.521299   
4193  0.01.01  35903.50525  129713.674366  129733.859866  129693.488866   
4197  0.01.01  35909.80525  129717.641934  129737.827434  129697.456434   
4201  0.01.01  35916.10525  129721.609502  129741.795002  129701.424002   
4206  0.01.01  35922.40525  129725.577070  129745.762570  129705.391570   
4211  0.01.01  35928.70525  129725.577070  129745.762570  129705.391570   
4216  0.01.01  35935.00525  129725.577070  129745.762570  129705.391570   
4222  0.01.01  35941.30525  129730.576488  129750.761988  129710.390988   
4227  0.01.01  35947.60525  129730.576488  129750.761988  129710.390988   
4233  0.01.01  35953.9052

Breaking the trenches into segments to indicate how many strings there are in each one.
=

In [103]:
"""
Adds cumulative sum column where values sum towards the inverter_line, calculated separately for each distinct group/segment.
"""
groups_df['STR_CUM_TO_INV'] = 0

# Group by GRP
# Each GRP represents a distinct physical segment with its own inverter
for grp_name in groups_df['GRP'].unique():
    # Get rows belonging to this GRP
    grp_mask = groups_df['GRP'] == grp_name
    grp_indices = groups_df[grp_mask].index

    if len(grp_indices) == 0:
        continue

    # Get the inverter_line for this GRP (should be consistent within group)
    # Using the first occurrence's inverter_line as the target for this group
    inv_line = groups_df.loc[grp_indices[0], 'inverter_line']

    # Separate rows by position relative to inverter_line
    before_inv = []
    at_inv = []
    after_inv = []

    for idx in grp_indices:
        x_val = groups_df.loc[idx, 'Line']
        if x_val < inv_line:
            before_inv.append(idx)
        elif x_val > inv_line:
            after_inv.append(idx)
        else:  # x_val == inv_line
            at_inv.append(idx)

    # Sort before_inv by X ascending (farthest to closest to inverter)
    before_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'])

    # Sort after_inv by X descending (farthest to closest to inverter)
    after_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'], reverse=True)

    # Calculate cumulative sums for before inverter (left to right)
    cumsum = 0
    for idx in before_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum

    # At inverter line: just the value itself
    for idx in at_inv:
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = groups_df.loc[idx, 'No. of Strings']

    # Calculate cumulative sums for after inverter (right to left)
    cumsum = 0
    for idx in after_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum


# Verify the results
print(groups_df[['GRP', 'X', 'No. of Strings', 'inverter_line', 'STR_CUM_TO_INV']].to_string())
max_number_of_strings = groups_df['STR_CUM_TO_INV'].max()
print(groups_df['STR_CUM_TO_INV'].value_counts())
groups_df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")

          GRP             X  No. of Strings  inverter_line  STR_CUM_TO_INV
4173  0.01.01  35878.305250               1        35948.0               1
4178  0.01.01  35884.605250               1        35948.0               2
4183  0.01.01  35890.905250               1        35948.0               3
4189  0.01.01  35897.205250               1        35948.0               4
4193  0.01.01  35903.505250               1        35948.0               5
4197  0.01.01  35909.805250               1        35948.0               6
4201  0.01.01  35916.105250               1        35948.0               7
4206  0.01.01  35922.405250               1        35948.0               8
4211  0.01.01  35928.705250               1        35948.0               9
4216  0.01.01  35935.005250               1        35948.0              10
4222  0.01.01  35941.305250               1        35948.0              11
4227  0.01.01  35947.605250               1        35948.0               1
4233  0.01.01  35953.9052

In [104]:
distance2 = 0.8805 # Distance between the first/last pile and the first/last module

cables_df.drop(columns=['UFV_x', 'UFV_y', 'PST_x', 'PST_y', 'INV_x', 'INV_y', 'SubGroup', 'Group'], inplace=True)
cables_df.rename(columns={'Position_X': 'X', 'Position_Y': 'Y', 'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
cables_df['N or S or M'] = np.where(
    abs(cables_df['N exit'] - cables_df['inverter_y']) > abs(cables_df['S exit'] - cables_df['inverter_y']),
    "S",
    np.where(
        (cables_df['inverter_y'] < cables_df['N exit']) & (cables_df['inverter_y'] > cables_df['S exit']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
cables_df['closest_y'] = np.where(
    abs(cables_df['Y-N-TRK'] - cables_df['inverter_y']) < abs(cables_df['Y'] - cables_df['inverter_y']),
    cables_df['Y-N-TRK'],
    np.where(
        abs(cables_df['Y'] - cables_df['inverter_y']) < abs(cables_df['Y-S-TRK'] - cables_df['inverter_y']),
        cables_df['Y'],
        cables_df['Y-S-TRK']
    )
)


cables_df['Y-N-STR'] = cables_df['Y-N-STR'] + distance2
cables_df['Y-S-STR'] = cables_df['Y-S-STR'] - distance2

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

        name            X              Y      GRP     Line         N exit  \
11159  1x32p  35878.30525  129697.804096  0.01.01  35878.0  129717.989596   
11168  1x32p  35884.60525  129701.771663  0.01.01  35885.0  129721.957163   
11178  1x32p  35890.90525  129705.739231  0.01.01  35891.0  129725.924731   
11187  1x32p  35897.20525  129709.706799  0.01.01  35897.0  129729.892299   
11195  1x32p  35903.50525  129713.674366  0.01.01  35904.0  129733.859866   

              S exit        Y-N-STR        Y-S-STR        Y-N-TRK  \
11159  129677.618596  129718.870096  129676.738096  129717.989596   
11168  129681.586163  129722.837663  129680.705663  129721.957163   
11178  129685.553731  129726.805231  129684.673231  129725.924731   
11187  129689.521299  129730.772799  129688.640799  129729.892299   
11195  129693.488866  129734.740366  129692.608366  129733.859866   

             Y-S-TRK    inverter_x     inverter_y QBT DSJ PVM  inverter_line  \
11159  129677.618596  35947.582968  129708

In [105]:
script_df = groups_df[['GRP','No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_y', 'X', 'inverter_x', 'inverter_y','Line', 'inverter_line', 'STR_CUM_TO_INV', 'Y-N-STR', 'Y-S-STR', 'N or S or M']].copy()
script_df.rename(columns={'closest_y': 'Y'}, inplace=True)

distance3 = 2.516 # Vertical distance between the tracker's pile and the inverter.
distance4 = 0.5325 # Distance between the middle point of the inverter and its edge on the X axis.
distance5 = 0.2 # Distance between the middle point of the inverter and its edge on the Y axis.
distance6 = 1 # Distance to make an intermediary point for stretching purposes.

def intermediary_coordinates(df):
    df['intermediary_x_1'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['X'] + distance3,
            df['X'] - distance3
        )
    )

    df['intermediary_x_2'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance6,
            df['inverter_x'] + distance6
        )
    )

    df['inverter_x'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance4,
            df['inverter_x'] + distance4
        )
    )

    df['inverter_y'] = np.where(
        df['inverter_line'] != df['Line'],
        df['inverter_y'],
        np.where(
            df['N or S or M'] == 'S',
            df['inverter_y'] + distance5,
            df['inverter_y'] - distance5
        )
    )

'''

CABOS

'''

intermediary_coordinates(cables_df)

def calculate_cable_length(row, color):
    try:
        x = row['X']
        y_n_str = row['N exit']
        y_s_str = row['S exit']
        closest_y = row['closest_y']
        int_x_1 = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']

        if color == 'red':
            seg1 = math.sqrt((x - x)**2 + (y_n_str - closest_y)**2)
        else:
            seg1 = math.sqrt((x - x)**2 + (y_s_str - closest_y)**2)
        seg2 = math.sqrt((x - int_x_1)**2 + (closest_y - inv_y)**2)
        seg3 = math.sqrt((int_x_1 - inv_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2 + seg3
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

cables_df['Red Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'red'), axis=1)
cables_df['Black Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'black'), axis=1)

total_red_length = cables_df['Red Cable Length'].sum()
total_black_length = cables_df['Black Cable Length'].sum()

total_cable_length = total_red_length + total_black_length

cables_df['Script Red Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['N exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
cables_df['Script Black Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['S exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)

with open (r"C:\Users\Usuario\Desktop\paiva\Script Cabos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Red Cables"].astype(str)) + '\n')
    f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    f.write('\n'.join(cables_df["Script Black Cables"].astype(str)) + '\n')

'''

VALAS

'''

intermediary_coordinates(script_df)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\paiva\Script Valas Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_bt_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''

DUTOS

'''

def calculate_duct_length(row, no_of_strings):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']
        number_of_ducts = row['Duct Definition'].count(no_of_strings)

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - inv_x)**2 + (inv_y - inv_y)**2)

        return (seg1 + seg2) * number_of_ducts
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

script_df['Dutos 1 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '1'), axis=1)
script_df['Dutos 2 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '2'), axis=1)
script_df['Dutos 3 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '3'), axis=1)
script_df['Dutos 4 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '4'), axis=1)

total_1str_duct_length = script_df['Dutos 1 Length'].sum()
total_2str_duct_length = script_df['Dutos 2 Length'].sum()
total_3str_duct_length = script_df['Dutos 3 Length'].sum()
total_4str_duct_length = script_df['Dutos 4 Length'].sum()

total_duct_length = total_1str_duct_length + total_2str_duct_length + total_3str_duct_length + total_4str_duct_length

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: '\n'.join([
        f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} "
        for _ in range(str(row['Duct Definition']).count('3'))
    ]),
    axis=1
)
script_df['Script Dutos 4'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('4'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\paiva\Script Dutos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write('\n'.join(layer3_commands) + '\n')

    # Layer 4
    layer4_commands = script_df["Script Dutos 4"].astype(str)
    layer4_commands = layer4_commands[layer4_commands != '']
    if len(layer4_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_4S\n\n")
        f.write('\n'.join(layer4_commands) + '\n')

4173    _.pline 35878.30525000301,129717.98959553531 3...
4178    _.pline 35884.60525000302,129701.7716632685 35...
4183    _.pline 35890.90525000303,129705.7392310017 35...
4189    _.pline 35897.20525000304,129709.706798735 358...
4193    _.pline 35903.50525000305,129713.6743664682 35...
                              ...                        
573     _.pline 34387.48615000009,125998.614 34384.970...
587     _.pline 34393.7861500001,125998.614 34391.2701...
601     _.pline 34400.08615000009,125998.614 34397.570...
615     _.pline 34406.38615000007,125998.614 34403.870...
630     _.pline 34412.68615000008,125998.614 34410.170...
Name: Script Valas, Length: 4674, dtype: str


In [106]:
'''

LAYERED VALAS

'''

pitch = 6.3

script_df['intermediary_x_3'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df['intermediary_x_1'] + pitch,
        script_df['intermediary_x_1'] - pitch
    )
)

script_df['final_x'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df.apply(lambda row: min(row['intermediary_x_3'], row['inverter_x']), axis=1),
        script_df.apply(lambda row: max(row['intermediary_x_3'], row['inverter_x']), axis=1)
    )
)

def calculate_trench_length(row):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        final_x = row['final_x']

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - final_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Valas Length'] = script_df.apply(calculate_trench_length, axis=1)

total_trench_length = script_df['Valas Length'].sum()

script_df['Script Valas Layered 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} ",
    axis=1
)

script_df['Script Valas Layered 2'] = script_df.apply(
    lambda row: f"_.pline {row['intermediary_x_1']},{row['inverter_y']} {row['final_x']},{row['inverter_y']} ",
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\paiva\Script Valas Layered Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    for i in range(1, max_number_of_strings+1):
        mask1 = script_df['No. of Strings'] == i
        mask2 = script_df['STR_CUM_TO_INV'] == i

        if mask1.any() or mask2.any():
            f.write(f"LAYER S EMF_vala_CC_{i}C\n\n")

            if mask1.any():
                f.write('\n'.join(script_df[mask1]["Script Valas Layered 1"].astype(str)) + '\n')

            if mask2.any():
                f.write('\n'.join(script_df[mask2]["Script Valas Layered 2"].astype(str)) + '\n')


lengths_df = pd.DataFrame.from_dict(data={
    'DC Red Cable Length': [total_red_length],
    'DC Black Cable Length': [total_black_length],
    'DC Total Cable Length': [total_cable_length],
    '1str Duct Length': [total_1str_duct_length],
    '2str Duct Length': [total_2str_duct_length],
    '3str Duct Length': [total_3str_duct_length],
    '4str Duct Length': [total_4str_duct_length],
    'Total Duct Length': [total_duct_length],
    'Total Trench Length': [total_trench_length]
}, orient="index", columns=['Length (m)'])
lengths_df.to_excel(r"C:\Users\Usuario\Desktop\Lengths.xlsx")
script_df.to_excel(r"C:\Users\Usuario\Desktop\SCRIPTS.xlsx")
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")